> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Founder probability kernels

VIGOR 2.12.1. Founder probabilities are optional; SNP dosages cannot substitute for founder ancestry.


In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__


In [ ]:
haplotypes=ROOT/"founder_probabilities.zarr"
if haplotypes.exists():
    hp=xr.open_zarr(haplotypes,consolidated=True,chunks={})
    # Slice before computing: this example deliberately fits a bounded ancestry panel.
    P=hp["HP"].isel(variant=slice(0,128)).compute().values
    K=vigor.founder_kernel(P)
    assert np.isfinite(K).all()
    print("Founder kernel:",K.shape)
else:print("Founder probabilities unavailable. Provide founder_probabilities.zarr with HP(sample,variant,founder).")
